# V17: Multi-GNN — Full PP Topology Comparison: None vs Jaccard vs BA (Scale-Free)

**New in V17 over V16:**
- **3 PP graph topologies compared head-to-head** for every model × pipeline combination:
  - `none` — bipartite Gene-Patient only, no Patient-Patient edges
  - `jaccard` — Patient-Patient edges based on observed mutation similarity (top-k Jaccard)
  - `ba` — Barabási-Albert scale-free PP layer (V16 default)
- **Total experiments**: 7 models × 4 pipelines × 3 topologies = **84 combinations**
- **New graph builder**: `construct_jaccard_heterograph()` — edges grounded in real biological similarity
- **Topology-aware HPO**: ba_m tuned when topology=ba; top_k tuned when topology=jaccard
- **All results tagged** with `Graph_Topology` column — flows into all tables, heatmaps, ROC plots
- **New visualisation**: AUC heatmap per topology, topology comparison bar chart per model
- All V16 improvements retained: fixed RGCN/SGNN, 7-dim features, focal loss, G-mean threshold


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import random, copy, warnings

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report,
                             precision_score, recall_score, f1_score, accuracy_score)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold

from torch_geometric.data import HeteroData
from torch_geometric.nn import (GATv2Conv, GCNConv, Linear, HGTConv,
                                 RGCNConv, HypergraphConv, ChebConv)

from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer
from sdv.sampling import Condition
from imblearn.over_sampling import SMOTENC
from torch.optim.lr_scheduler import CosineAnnealingLR

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

import os

def set_seed(seed=42):
    """Full determinism:
    - Python / NumPy / PyTorch RNGs
    - cuDNN deterministic algorithms (no non-deterministic scatter ops)
    - PYTHONHASHSEED for dict/set ordering
    NOTE: cudnn.deterministic=True adds ~10-20% training overhead.
    """
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True   # no non-det cuDNN kernels
    torch.backends.cudnn.benchmark     = False  # disable auto-tuner (picks diff algo each run)

set_seed(42)

N_TRIALS   = 30    # Optuna TPE trials per model×pipeline
N_FOLDS    = 5     # stratified CV folds
MAX_EPOCHS = 200
PATIENCE   = 20

/home/nafim/anaconda3/envs/thesis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/nafim/anaconda3/envs/thesis/lib/python3.11/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: libnvToolsExt.so.1: cannot open shared object file: No such file or directory
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/home/nafim/anaconda3/envs/thesis/lib/python3.11/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/nafim/anaconda3/envs/thesis/lib/python3.11/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "


Using device: cuda
GPU: NVIDIA GeForce RTX 3050


## 2. Load Data

In [2]:
tcga_df = pd.read_csv('../../dataset/TCGA_InfoWithGrade.csv')
cgga_df = pd.read_csv('../../dataset/weseq_processed_with_id_and_race_V2.csv')

if 'CGGA_ID' in cgga_df.columns:
    cgga_df = cgga_df.drop(columns='CGGA_ID')

categorical_columns = ['Grade', 'Gender', 'Race']
gene_columns = ['IDH1','TP53','ATRX','PTEN','EGFR','CIC','MUC16','PIK3CA',
                'NF1','PIK3R1','FUBP1','RB1','NOTCH1','BCOR','CSMD3',
                'SMARCA4','GRIN2A','IDH2','FAT4','PDGFRA']
NUM_GENES = len(gene_columns)

print("TCGA:", tcga_df.shape, "  Grade dist:", dict(tcga_df['Grade'].value_counts()))
print("CGGA:", cgga_df.shape, "  Grade dist:", dict(cgga_df['Grade'].value_counts()))

TCGA: (839, 24)   Grade dist: {0: 487, 1: 352}
CGGA: (286, 24)   Grade dist: {0: 184, 1: 102}


## 3. Splits — 20% held-out test; remaining 80% used for HPO + 5-Fold CV

In [3]:
# Hold out 20% as final test set (never seen during HPO or CV)
train_val_df, test_df = train_test_split(
    tcga_df, test_size=0.2, stratify=tcga_df['Grade'], random_state=42)

# Further split train_val for Optuna HPO warmup (val graph used only in HPO phase)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.2, stratify=train_val_df['Grade'], random_state=42)

print(f"Train(HPO)={len(train_df)}  Val(HPO)={len(val_df)}  "
      f"TrainVal(CV)={len(train_val_df)}  Test={len(test_df)}")
print("Train Grade dist:", dict(train_df['Grade'].value_counts()))
print("Test  Grade dist:", dict(test_df['Grade'].value_counts()))

Train(HPO)=536  Val(HPO)=135  TrainVal(CV)=671  Test=168
Train Grade dist: {0: 311, 1: 225}
Test  Grade dist: {0: 98, 1: 70}


In [4]:
# ── Cell 5b — Fit normalisation scalers on train_df ONLY ─────────
# This must run immediately after the train/val/test split and before
# any graph is built. All calls to _build_patient_features() will
# use these scalers via .transform() (never .fit()) on any split.

def fit_scalers_on_train(df):
    global _age_scaler, _burden_scaler
    _age_scaler    = StandardScaler().fit(df[['Age_at_diagnosis']])
    _burden_scaler = StandardScaler().fit(
        df[gene_columns].sum(axis=1).values.reshape(-1,1).astype('float32'))
    print(f'Age scaler    — mean={_age_scaler.mean_[0]:.2f}  '
          f'std={_age_scaler.scale_[0]:.2f}')
    print(f'Burden scaler — mean={_burden_scaler.mean_[0]:.2f}  '
          f'std={_burden_scaler.scale_[0]:.2f}')
    print(f'Fitted on {len(df)} training patients (train_df only — no leakage).')

fit_scalers_on_train(train_df)


Age scaler    — mean=51.22  std=15.36
Burden scaler — mean=2.27  std=1.42
Fitted on 536 training patients (train_df only — no leakage).


## 4. Graph Construction

In [5]:
_PP_CACHE: dict = {}

RTK_GENES  = ['EGFR', 'PDGFRA', 'PIK3CA', 'PIK3R1', 'PTEN', 'NF1']
IDH_GENES  = ['IDH1', 'IDH2']
TP53_GENES = ['TP53', 'ATRX', 'RB1']
PAT_FEAT_DIM = 7

# ── V17 FIX 1: Global scalers fitted ONCE on train_df (Cell 5b) ──
# V16 bug: StandardScaler().fit_transform(df) was called on every df
# including val/test/CGGA — each got its OWN normalisation statistics.
# Fix: .fit() only in Cell 5b on train_df; all graph builds call .transform().
_age_scaler    = None
_burden_scaler = None


def _build_patient_features(df):
    """7-dim patient features — uses train-fitted scalers, no data leakage."""
    import warnings; warnings.filterwarnings('ignore')
    if _age_scaler is None or _burden_scaler is None:
        raise RuntimeError('Run Cell 5b to fit scalers before building any graph.')
    age_norm   = _age_scaler.transform(
        df[['Age_at_diagnosis']]).astype('float32')
    mut_burden = _burden_scaler.transform(
        df[gene_columns].sum(axis=1).values.reshape(-1,1).astype('float32'))
    rtk_cols   = [g for g in RTK_GENES  if g in df.columns]
    idh_cols   = [g for g in IDH_GENES  if g in df.columns]
    tp53_cols  = [g for g in TP53_GENES if g in df.columns]
    rtk_score  = df[rtk_cols ].sum(1).values.reshape(-1,1).astype('float32') if rtk_cols  else np.zeros((len(df),1),'float32')
    idh_score  = df[idh_cols ].sum(1).values.reshape(-1,1).astype('float32') if idh_cols  else np.zeros((len(df),1),'float32')
    tp53_score = df[tp53_cols].sum(1).values.reshape(-1,1).astype('float32') if tp53_cols else np.zeros((len(df),1),'float32')
    return np.hstack([df[['Gender','Race']].values.astype('float32'),
                      age_norm, mut_burden, rtk_score, idh_score, tp53_score])


def construct_bipartite_heterograph(df):
    """Used ONLY for scale-free degree analysis in Cell 9.
    NOT for model training — use build_graph() for all model graphs."""
    graph = HeteroData()
    graph['Patient'].x = torch.tensor(_build_patient_features(df), dtype=torch.float)
    graph['Patient'].y = torch.tensor(df['Grade'].values, dtype=torch.long)
    graph['Gene'].x    = torch.eye(NUM_GENES, dtype=torch.float)
    src_genes, dst_patients = [], []
    for p_idx, (_, row) in enumerate(df.iterrows()):
        for g_idx, gene in enumerate(gene_columns):
            if int(row[gene]) == 1:
                src_genes.append(g_idx); dst_patients.append(p_idx)
    graph['Gene','mutates','Patient'].edge_index    = torch.tensor([src_genes,    dst_patients], dtype=torch.long)
    graph['Patient','mutated_by','Gene'].edge_index = torch.tensor([dst_patients, src_genes],    dtype=torch.long)
    # V17 FIX 4: always include PP key so models never get a KeyError
    graph[('Patient','cooccurs','Patient')].edge_index = torch.zeros(2, 0, dtype=torch.long)
    return graph


# V17: get_pp_edges() removed — it was dead code. No model ever called it.
# All models read PP edges directly from graph[('Patient','cooccurs','Patient')].

def to_dev(graph): return graph.to(device)
def clear_pp_cache(): _PP_CACHE.clear()

print(f'PAT_FEAT_DIM={PAT_FEAT_DIM}')
print('Scalers not fitted yet — run Cell 5b next.')


PAT_FEAT_DIM=7
Scalers not fitted yet — run Cell 5b next.


## 4a. Scale-Free Analysis — Degree Distribution Proof

In [6]:
from scipy import stats

def get_degree_sequences(graph):
    ei       = graph[("Gene","mutates","Patient")].edge_index.cpu()
    gene_ids = ei[0].numpy(); pat_ids = ei[1].numpy()
    pat_deg  = np.bincount(pat_ids,  minlength=graph["Patient"].x.shape[0])
    gene_deg = np.bincount(gene_ids, minlength=graph["Gene"].x.shape[0])
    return pat_deg, gene_deg


def fit_powerlaw(degrees, label="Degrees"):
    k_vals, counts = np.unique(degrees[degrees > 0], return_counts=True)
    pmf   = counts / len(degrees[degrees > 0])
    log_k = np.log(k_vals.astype(float))
    log_p = np.log(pmf + 1e-12)

    sl_pl,  ic_pl,  r_pl,  *_ = stats.linregress(log_k, log_p)
    sl_exp, ic_exp, r_exp, *_ = stats.linregress(k_vals.astype(float), log_p)
    gamma  = -sl_pl
    r2_pl  = r_pl  ** 2
    r2_exp = r_exp ** 2

    fitted_pl = np.exp(ic_pl) * (k_vals ** sl_pl)
    fitted_pl = fitted_pl / fitted_pl.sum()
    _, ks_p   = stats.ks_2samp(pmf, fitted_pl)

    print(f"\n{chr(8212)*55}\n  {label}\n{chr(8212)*55}")
    print(f"  Degree range       : [{degrees.min()}, {degrees.max()}]")
    print(f"  Mean +/- Std       : {degrees.mean():.2f} +/- {degrees.std():.2f}")
    print(f"  Power-law gamma    : {gamma:.3f}  (scale-free: 2 < gamma < 3)")
    print(f"  R2 power-law       : {r2_pl:.4f}")
    print(f"  R2 exponential     : {r2_exp:.4f}")
    print(f"  KS p-value (PL)    : {ks_p:.4f}  (p>0.05 = good PL fit)")

    is_sf = r2_pl > r2_exp and 2 < gamma < 3 and ks_p > 0.05
    if is_sf:
        print("  Verdict            : SCALE-FREE")
    else:
        reasons = []
        if r2_pl <= r2_exp:  reasons.append(f"exp fits better (R2_exp={r2_exp:.3f} > R2_pl={r2_pl:.3f})")
        if not (2<gamma<3):  reasons.append(f"gamma={gamma:.2f} outside (2,3)")
        if ks_p <= 0.05:     reasons.append(f"KS rejects PL (p={ks_p:.4f})")
        print(f"  Reason(s)          : {'; '.join(reasons)}")
        print("  Verdict            : NOT SCALE-FREE")
    return gamma, r2_pl, r2_exp, ks_p


def plot_degree_distribution(pat_deg, gene_deg, title_prefix="Graph"):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, degs, lbl in zip(
        axes,
        [pat_deg, gene_deg],
        ["Patient degree (# mutations per patient)",
         "Gene degree (# patients with mutation)"]
    ):
        k_vals, counts = np.unique(degs[degs > 0], return_counts=True)
        pmf = counts / len(degs[degs > 0])
        ax.loglog(k_vals, pmf, "o", color="steelblue", ms=6, label="Empirical P(k)")
        log_k, log_p = np.log(k_vals.astype(float)), np.log(pmf + 1e-12)
        sl_pl, ic_pl, *_ = stats.linregress(log_k, log_p)
        ax.loglog(k_vals, np.exp(ic_pl + sl_pl * log_k), "r--", lw=2,
                  label=f"Power-law  gamma={-sl_pl:.2f}")
        sl_ex, ic_ex, *_ = stats.linregress(k_vals.astype(float), log_p)
        ax.loglog(k_vals, np.exp(ic_ex + sl_ex * k_vals.astype(float)), "g:", lw=2,
                  label="Exponential")
        ax.set_xlabel("Degree k"); ax.set_ylabel("P(k)")
        ax.set_title(f"{title_prefix}\n{lbl}")
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
        ax.text(0.97, 0.97, "Scale-free => straight line on log-log",
                transform=ax.transAxes, fontsize=7.5, va="top", ha="right", color="gray",
                bbox=dict(fc="white", alpha=0.7, ec="lightgray"))
    plt.suptitle(f"Degree Distribution - {title_prefix}", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig("V15_scalefree_degree_dist.png", dpi=150, bbox_inches="tight")
    plt.show()


# ── Run analysis ───────────────────────────────────────────────────────────────
print("=" * 65)
print("SCALE-FREE ANALYSIS - Original Bipartite Heterogeneous Graph")
print("=" * 65)
print("""
  Graph type  : Bipartite HeteroData  (Gene <-> Patient)
  Patient nodes : [Gender, Race, Age_norm]  (N ~800+)
  Gene nodes    : one-hot identity          (N = 20)
  Edges         : Gene->Patient if mutation bit == 1
""")

_ag = to_dev(construct_bipartite_heterograph(train_val_df))
_pd, _gd = get_degree_sequences(_ag)
_g_p, _r2pl_p, _r2ex_p, _ks_p = fit_powerlaw(_pd,  "Patient-side degrees")
_g_g, _r2pl_g, _r2ex_g, _ks_g = fit_powerlaw(_gd, "Gene-side degrees")

print(f"""
FORMAL VERDICT
--------------
Scale-free iff P(k) ~ k^(-gamma), 2<gamma<3, R2_PL > R2_exp

Patient side : gamma={_g_p:.2f}  R2_PL={_r2pl_p:.3f}  R2_exp={_r2ex_p:.3f}
Gene side    : gamma={_g_g:.2f}  R2_PL={_r2pl_g:.3f}  R2_exp={_r2ex_g:.3f}

WHY IT FAILS:
  1. Patient degree hard-capped at NUM_GENES=20 -> no heavy tail
  2. Gene frequencies Poisson-like, not power-law
  3. No preferential attachment (static mutation matrix)

-> APPLYING BA REWIRING BELOW
""")
plot_degree_distribution(_pd, _gd, title_prefix="V12 Original Graph")


SCALE-FREE ANALYSIS - Original Bipartite Heterogeneous Graph

  Graph type  : Bipartite HeteroData  (Gene <-> Patient)
  Patient nodes : [Gender, Race, Age_norm]  (N ~800+)
  Gene nodes    : one-hot identity          (N = 20)
  Edges         : Gene->Patient if mutation bit == 1



RuntimeError: Run Cell 5b to fit scalers before building any graph.

## 4b. Three PP Graph Topologies

V17 compares three strategies for the Patient-Patient (PP) layer:

| Topology | How edges are formed | Biological justification | Key parameter |
|---|---|---|---|
| `none` | No PP edges — bipartite only | Patients share info only through shared genes (2-hop) | — |
| `jaccard` | Top-k most similar patients by Jaccard mutation similarity | Directly observed molecular similarity | `top_k` |
| `ba` | Barabási-Albert preferential attachment seeded from co-mutations | Scale-free topology prior | `ba_m` |

Each topology produces a different `('Patient','cooccurs','Patient')` edge set. All other graph components (bipartite edges, node features) are identical.


In [ ]:
# ── Topology 1: No PP layer (bipartite only) ────────────────────
def construct_bipartite_only_heterograph(df):
    """Bipartite Gene-Patient graph with NO Patient-Patient edges.
    Patients share information only through shared gene nodes (2-hop paths).
    This is the baseline — tests whether the PP layer adds any value at all.
    """
    graph = HeteroData()
    graph['Patient'].x = torch.tensor(_build_patient_features(df), dtype=torch.float)
    graph['Patient'].y = torch.tensor(df['Grade'].values, dtype=torch.long)
    graph['Gene'].x    = torch.eye(NUM_GENES, dtype=torch.float)
    src_genes, dst_pats = [], []
    mut_mat = df[gene_columns].values.astype(int)
    for p_idx, row in enumerate(mut_mat):
        for g_idx in np.where(row==1)[0]:
            src_genes.append(g_idx); dst_pats.append(p_idx)
    graph[('Gene','mutates','Patient')].edge_index    = torch.tensor([src_genes, dst_pats], dtype=torch.long)
    graph[('Patient','mutated_by','Gene')].edge_index = torch.tensor([dst_pats, src_genes], dtype=torch.long)
    # Empty PP layer — models handle this gracefully (zero-contribution convolution)
    graph[('Patient','cooccurs','Patient')].edge_index = torch.zeros(2, 0, dtype=torch.long)
    return graph


# ── Topology 2: Jaccard similarity graph ─────────────────────────
def construct_jaccard_heterograph(df, top_k=5):
    """Patient-Patient edges based on Jaccard similarity of mutation profiles.

    Jaccard(A,B) = |mutations_A ∩ mutations_B| / |mutations_A ∪ mutations_B|

    Each patient is connected to their top_k most similar patients.
    This is biologically grounded — edges reflect OBSERVED molecular similarity,
    not a synthetic random process. Two patients are connected because they
    actually share mutations, not because of preferential attachment.

    top_k: number of nearest neighbours per patient (tuned by Optuna)
    """
    graph = HeteroData()
    graph['Patient'].x = torch.tensor(_build_patient_features(df), dtype=torch.float)
    graph['Patient'].y = torch.tensor(df['Grade'].values, dtype=torch.long)
    graph['Gene'].x    = torch.eye(NUM_GENES, dtype=torch.float)
    src_genes, dst_pats = [], []
    mut_mat = df[gene_columns].values.astype(int).astype(float)
    n_p = len(df)
    for p_idx, row in enumerate(mut_mat):
        for g_idx in np.where(row==1)[0]:
            src_genes.append(g_idx); dst_pats.append(p_idx)
    graph[('Gene','mutates','Patient')].edge_index    = torch.tensor([src_genes, dst_pats], dtype=torch.long)
    graph[('Patient','mutated_by','Gene')].edge_index = torch.tensor([dst_pats, src_genes], dtype=torch.long)

    # ── Build Jaccard PP edges ────────────────────────────────────
    # Vectorised: intersection = mut_mat @ mut_mat.T
    #             union        = (row_sums broadcast) + (row_sums.T broadcast) - intersection
    inter = mut_mat @ mut_mat.T                              # (n_p, n_p)
    row_s = mut_mat.sum(axis=1, keepdims=True)               # (n_p, 1)
    union = row_s + row_s.T - inter                          # (n_p, n_p)
    union = np.maximum(union, 1e-9)                          # avoid div-by-zero
    jac   = inter / union                                    # (n_p, n_p)
    np.fill_diagonal(jac, 0.0)                               # no self-loops

    pp_src, pp_dst = [], []
    for i in range(n_p):
        # Top-k neighbours by Jaccard similarity
        scores = jac[i]
        top_j  = np.argsort(scores)[::-1][:top_k]
        for j in top_j:
            if scores[j] > 0:   # only connect if any shared mutation
                pp_src += [i, j]; pp_dst += [j, i]

    pp_edges = list({(s,d) for s,d in zip(pp_src,pp_dst) if s!=d})
    if pp_edges:
        ps, pd_ = zip(*pp_edges)
        graph[('Patient','cooccurs','Patient')].edge_index = torch.tensor([list(ps),list(pd_)], dtype=torch.long)
    else:
        graph[('Patient','cooccurs','Patient')].edge_index = torch.zeros(2,0,dtype=torch.long)
    return graph


# ── Topology 3: Barabási-Albert scale-free (V16 default) ─────────
def construct_scalefree_bipartite_heterograph(df, ba_m=2, seed=42):
    """BA scale-free PP layer seeded from co-mutation degree."""
    rng = np.random.default_rng(seed)
    graph = HeteroData()
    graph['Patient'].x = torch.tensor(_build_patient_features(df), dtype=torch.float)
    graph['Patient'].y = torch.tensor(df['Grade'].values, dtype=torch.long)
    graph['Gene'].x    = torch.eye(NUM_GENES, dtype=torch.float)
    src_genes, dst_pats = [], []
    mut_mat = df[gene_columns].values.astype(int)
    for p_idx, row in enumerate(mut_mat):
        for g_idx in np.where(row==1)[0]:
            src_genes.append(g_idx); dst_pats.append(p_idx)
    graph[('Gene','mutates','Patient')].edge_index    = torch.tensor([src_genes, dst_pats], dtype=torch.long)
    graph[('Patient','mutated_by','Gene')].edge_index = torch.tensor([dst_pats, src_genes], dtype=torch.long)
    n_p    = len(df)
    degree = np.zeros(n_p, dtype=float)
    for g in range(NUM_GENES):
        carriers = np.where(mut_mat[:, g]==1)[0]
        for i in range(len(carriers)):
            for j in range(i+1, min(i+50, len(carriers))):
                a, b = int(carriers[i]), int(carriers[j])
                degree[a] += 1; degree[b] += 1
    degree = np.maximum(degree, 1.0)
    ba_src, ba_dst = [], []
    for p in range(n_p):
        pool  = np.delete(np.arange(n_p), p)
        probs = degree[pool] / degree[pool].sum()
        chosen = rng.choice(pool, size=min(ba_m, len(pool)), replace=False, p=probs)
        for c in chosen:
            ba_src += [p, int(c)]; ba_dst += [int(c), p]
            degree[p] += 1; degree[int(c)] += 1
    pp_edges = list({(s,d) for s,d in zip(ba_src,ba_dst) if s!=d})
    if pp_edges:
        pp_s, pp_d = zip(*pp_edges)
        graph[('Patient','cooccurs','Patient')].edge_index = torch.tensor([list(pp_s),list(pp_d)], dtype=torch.long)
    else:
        graph[('Patient','cooccurs','Patient')].edge_index = torch.zeros(2,0,dtype=torch.long)
    return graph


# ── Unified graph builder dispatcher ─────────────────────────────
TOPOLOGY_NAMES = ['none', 'jaccard', 'ba']

def build_graph(df, topology='ba', ba_m=2, top_k=5):
    """Single entry point for all topology variants."""
    if topology == 'none':
        return construct_bipartite_only_heterograph(df)
    elif topology == 'jaccard':
        return construct_jaccard_heterograph(df, top_k=top_k)
    elif topology == 'ba':
        return construct_scalefree_bipartite_heterograph(df, ba_m=ba_m)
    else:
        raise ValueError(f'Unknown topology: {topology}')


# ── Quick verification of all three topologies ────────────────────
print('=' * 65)
print('V17: Verifying all 3 PP graph topologies on train_val_df')
print('=' * 65)
for _topo in TOPOLOGY_NAMES:
    _g = build_graph(train_val_df, topology=_topo, ba_m=2, top_k=5)
    _pp = _g[('Patient','cooccurs','Patient')].edge_index
    _n_edges = _pp.shape[1]
    _n_nodes = _g['Patient'].x.shape[0]
    _avg_deg = _n_edges / _n_nodes if _n_nodes > 0 else 0
    print(f'  [{_topo:8s}] PP edges={_n_edges:6d}  avg_degree={_avg_deg:.2f}  '
          f'pat_feat_dim={_g["Patient"].x.shape[1]}')
print('All topology builders OK.')


## 5. Build Shared Evaluation Graphs — One Per Topology

In [ ]:
# V17: Build separate val/test/cgga graphs for each topology.
# Models trained with topology X are always evaluated on topology X graphs.
# ba_m=2 and top_k=5 are fixed for eval graphs (HPO may tune them for training).

VAL_GRAPHS  = {t: to_dev(build_graph(val_df,  topology=t, ba_m=2, top_k=5)) for t in TOPOLOGY_NAMES}
TEST_GRAPHS = {t: to_dev(build_graph(test_df, topology=t, ba_m=2, top_k=5)) for t in TOPOLOGY_NAMES}
CGGA_GRAPHS = {t: to_dev(build_graph(cgga_df, topology=t, ba_m=2, top_k=5)) for t in TOPOLOGY_NAMES}

# Keep legacy names pointing to 'ba' for any code that uses them directly
val_graph  = VAL_GRAPHS['ba']
test_graph = TEST_GRAPHS['ba']
cgga_graph = CGGA_GRAPHS['ba']

print('Evaluation graphs built for all topologies:')
for t in TOPOLOGY_NAMES:
    n_pp = TEST_GRAPHS[t][('Patient','cooccurs','Patient')].edge_index.shape[1]
    print(f'  [{t:8s}] test PP edges={n_pp}')
print(f'Patient feature dim: {TEST_GRAPHS["ba"]["Patient"].x.shape[1]}')


## 6. Model Definitions (7 architectures)

| # | Model | Core idea |
|---|-------|-----------|
| 1 | **HeteroGATv2** | Bidirectional GATv2 on Gene-Patient bipartite graph |
| 2 | **MOGAT** | Dual genomic+clinical paths, soft fusion gate |
| 3 | **HyperTMO** | Genes = hyperedges, patients = nodes |
| 4 | **RGCN** | Patient-patient co-mutation graph, 20 gene-typed relations |
| 5 | **VEGN** | Learned per-edge variant-effect weights |
| 6 | **FastHGTConv** | Heterogeneous Graph Transformer |
| 7 | **SGNN** | Chebyshev spectral filters on patient co-mutation adjacency |

In [ ]:
# ── 1. HeteroGATv2 ─────────────────────────────────────────────
class HeteroGATv2(nn.Module):
    def __init__(self, hidden_dim=32, out_dim=2, num_heads=4, dropout=0.2, **_):
        super().__init__()
        self.dr = dropout
        self.p_lin = Linear(-1, hidden_dim); self.g_lin = Linear(-1, hidden_dim)
        self.g2p = GATv2Conv(hidden_dim, hidden_dim, heads=num_heads, concat=False, add_self_loops=False)
        self.p2g = GATv2Conv(hidden_dim, hidden_dim, heads=num_heads, concat=False, add_self_loops=False)
        self.p_skip = Linear(hidden_dim, hidden_dim); self.g_skip = Linear(hidden_dim, hidden_dim)
        self.pp_conv = GCNConv(hidden_dim, hidden_dim, add_self_loops=False)  # V17 FIX 3
        self.clf = Linear(hidden_dim, out_dim)
    def forward(self, graph):
        ei = graph.edge_index_dict
        hp = F.relu(self.p_lin(graph['Patient'].x))
        hg = F.relu(self.g_lin(graph['Gene'].x))
        hp = self.p_skip(hp) + self.g2p((hg,hp), ei[('Gene','mutates','Patient')])
        hg = self.g_skip(hg) + self.p2g((hp,hg), ei[('Patient','mutated_by','Gene')])
        pp_ei = graph[('Patient','cooccurs','Patient')].edge_index.to(hp.device)
        hp = hp + self.pp_conv(hp, pp_ei)
        return self.clf(F.dropout(F.leaky_relu(hp,0.2), self.dr, training=self.training))
    def get_attn_weights(self, graph):
        ei = graph.edge_index_dict
        hp = F.relu(self.p_lin(graph['Patient'].x))
        hg = F.relu(self.g_lin(graph['Gene'].x))
        _, (eidx, alpha) = self.g2p((hg,hp), ei[('Gene','mutates','Patient')],
                                     return_attention_weights=True)
        return eidx, alpha.detach().mean(dim=-1)


# ── 2. MOGAT ────────────────────────────────────────────────────
class MOGAT(nn.Module):
    def __init__(self, hidden_dim=32, out_dim=2, num_heads=4, dropout=0.2, **_):
        super().__init__()
        self.dr = dropout
        self.pg = Linear(-1, hidden_dim); self.gg = Linear(-1, hidden_dim)
        self.gat = GATv2Conv(hidden_dim, hidden_dim, heads=num_heads, concat=False, add_self_loops=False)
        self.pc  = Linear(-1, hidden_dim)
        self.mlp = nn.Sequential(nn.Linear(hidden_dim,hidden_dim), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden_dim,hidden_dim))
        self.gate    = nn.Linear(hidden_dim*2, 2)
        self.pp_conv = GCNConv(hidden_dim, hidden_dim, add_self_loops=False)  # V17 FIX 3
        self.clf     = Linear(hidden_dim, out_dim)
    def forward(self, graph):
        e   = graph[('Gene','mutates','Patient')].edge_index
        hpg = F.relu(self.pg(graph['Patient'].x)); hgg = F.relu(self.gg(graph['Gene'].x))
        hpg = F.dropout(F.leaky_relu(self.gat((hgg,hpg),e),0.2), self.dr, training=self.training)
        hpc = self.mlp(F.relu(self.pc(graph['Patient'].x)))
        g   = torch.softmax(self.gate(torch.cat([hpg,hpc],-1)), dim=-1)
        hp  = g[:,:1]*hpg + g[:,1:]*hpc
        pp_ei = graph[('Patient','cooccurs','Patient')].edge_index.to(hp.device)
        hp = hp + self.pp_conv(hp, pp_ei)
        return self.clf(hp)
    def get_attn_weights(self, graph):
        ei = graph.edge_index_dict
        hp = F.relu(self.pg(graph['Patient'].x))
        hg = F.relu(self.gg(graph['Gene'].x))
        _, (eidx, alpha) = self.gat((hg,hp), ei[('Gene','mutates','Patient')],
                                     return_attention_weights=True)
        return eidx, alpha.detach().mean(dim=-1)


# ── 3. HyperTMO ─────────────────────────────────────────────────
class HyperTMO(nn.Module):
    def __init__(self, hidden_dim=32, out_dim=2, num_heads=4, dropout=0.2, in_channels=PAT_FEAT_DIM, **_):
        super().__init__()
        self.dr = dropout
        self.p_lin = nn.Linear(in_channels, hidden_dim)
        self.g_lin = nn.Linear(NUM_GENES, hidden_dim)
        self.hc1   = HypergraphConv(hidden_dim, hidden_dim, use_attention=True,
                                    heads=num_heads, dropout=dropout, concat=False)
        self.hc2   = HypergraphConv(hidden_dim, hidden_dim)
        self.skip  = nn.Linear(hidden_dim, hidden_dim)
        self.bn    = nn.BatchNorm1d(hidden_dim)
        self.pp_conv = GCNConv(hidden_dim, hidden_dim, add_self_loops=False)  # V17 FIX 3
        self.clf   = nn.Linear(hidden_dim, out_dim)
    def forward(self, graph):
        xp = graph['Patient'].x; xg = graph['Gene'].x
        ei = graph[('Gene','mutates','Patient')].edge_index
        hei = torch.stack([ei[1], ei[0]], dim=0)
        hp = F.relu(self.p_lin(xp)); hg = F.relu(self.g_lin(xg)); s = self.skip(hp)
        hp = F.relu(self.hc1(hp, hei, hyperedge_attr=hg))
        hp = F.dropout(hp, self.dr, training=self.training)
        hp = self.bn(F.relu(self.hc2(hp,hei) + s))
        pp_ei = graph[('Patient','cooccurs','Patient')].edge_index.to(hp.device)
        hp = hp + self.pp_conv(hp, pp_ei)
        return self.clf(F.dropout(hp, self.dr, training=self.training))


# ── 4. RGCN — V16 FIX: Gene->Patient GATv2 aggregation added ────
# V15 bug: RGCN only used Patient.x (clinical features) + PP edges.
#          Gene mutation information was COMPLETELY IGNORED.
# V16 fix: GATv2Conv aggregates Gene->Patient FIRST, then RGCN on PP.
class RGCNModel(nn.Module):
    def __init__(self, hidden_dim=32, out_dim=2, num_heads=4, dropout=0.2,
                 in_channels=PAT_FEAT_DIM, num_relations=1, **_):  # V17 FIX 2
        super().__init__()
        self.dr = dropout
        self.p_lin  = nn.Linear(in_channels, hidden_dim)
        self.g_lin  = Linear(-1, hidden_dim)         # Gene encoder
        self.g2p    = GATv2Conv(hidden_dim, hidden_dim, heads=num_heads,
                                concat=False, add_self_loops=False)  # V16 FIX
        self.g2p_bn = nn.BatchNorm1d(hidden_dim)     # V16 FIX
        self.rc1    = RGCNConv(hidden_dim, hidden_dim, num_relations=num_relations)
        self.rc2    = RGCNConv(hidden_dim, hidden_dim, num_relations=num_relations)
        self.skip   = nn.Linear(hidden_dim, hidden_dim)
        self.bn     = nn.BatchNorm1d(hidden_dim)
        self.clf    = nn.Linear(hidden_dim, out_dim)
    def forward(self, graph):
        xp    = graph['Patient'].x;  xg = graph['Gene'].x
        e_g2p = graph[('Gene','mutates','Patient')].edge_index
        pp_ei = graph[('Patient','cooccurs','Patient')].edge_index.to(xp.device)
        pp_et = torch.zeros(pp_ei.shape[1], dtype=torch.long, device=xp.device)
        hp = F.relu(self.p_lin(xp))
        hg = F.relu(self.g_lin(xg))
        hp = self.g2p_bn(F.relu(hp + self.g2p((hg,hp), e_g2p)))  # V16 FIX
        s  = self.skip(hp)
        hp = F.relu(self.rc1(hp, pp_ei, pp_et))
        hp = F.dropout(hp, self.dr, training=self.training)
        hp = self.bn(F.relu(self.rc2(hp, pp_ei, pp_et) + s))
        return self.clf(F.dropout(hp, self.dr, training=self.training))


# ── 5. VEGN ─────────────────────────────────────────────────────
class VEGNModel(nn.Module):
    def __init__(self, hidden_dim=32, out_dim=2, num_heads=4, dropout=0.2, **_):
        super().__init__()
        self.dr = dropout
        self.p_lin = Linear(-1, hidden_dim); self.g_lin = Linear(-1, hidden_dim)
        self.ve  = nn.Sequential(nn.Linear(hidden_dim*2,hidden_dim), nn.ReLU(),
                                  nn.Linear(hidden_dim,1), nn.Sigmoid())
        self.p2g = GATv2Conv(hidden_dim, hidden_dim, heads=num_heads, concat=False, add_self_loops=False)
        self.skip = Linear(hidden_dim, hidden_dim)
        self.bn   = nn.BatchNorm1d(hidden_dim)
        self.pp_conv = GCNConv(hidden_dim, hidden_dim, add_self_loops=False)  # V17 FIX 3
        self.clf  = Linear(hidden_dim, out_dim)
    def forward(self, graph):
        xp = graph['Patient'].x; xg = graph['Gene'].x
        e_g2p = graph[('Gene','mutates','Patient')].edge_index
        e_p2g = graph[('Patient','mutated_by','Gene')].edge_index
        hp = F.relu(self.p_lin(xp)); hg = F.relu(self.g_lin(xg))
        sg, dp = e_g2p[0], e_g2p[1]
        wt  = self.ve(torch.cat([hg[sg], hp[dp]],-1)).squeeze(-1)
        msg = hg[sg] * wt.unsqueeze(-1)
        agg = torch.zeros_like(hp); agg.scatter_add_(0, dp.unsqueeze(-1).expand_as(msg), msg)
        deg = torch.zeros(hp.shape[0], device=hp.device)
        deg.scatter_add_(0, dp, wt); deg = deg.clamp(min=1)
        agg = agg / deg.unsqueeze(-1)
        _ = self.p2g((hp,hg), e_p2g)
        hp2 = self.bn(F.leaky_relu(self.skip(hp)+agg, 0.2))
        pp_ei = graph[('Patient','cooccurs','Patient')].edge_index.to(hp2.device)
        hp2 = hp2 + self.pp_conv(hp2, pp_ei)
        return self.clf(F.dropout(hp2, self.dr, training=self.training))


# ── 6. FastHGTConv ──────────────────────────────────────────────
_HGT_META_SF = (['Patient','Gene'],
                [('Gene','mutates','Patient'),('Patient','mutated_by','Gene'),
                 ('Patient','cooccurs','Patient')])

class FastHGTModel(nn.Module):
    def __init__(self, hidden_dim=32, out_dim=2, num_heads=4, dropout=0.2, hgt_meta=None, **_):
        super().__init__()
        self.dr   = dropout
        meta = hgt_meta if hgt_meta is not None else _HGT_META_SF
        self.p_lin = Linear(-1, hidden_dim); self.g_lin = Linear(-1, hidden_dim)
        self.hgt1  = HGTConv(hidden_dim, hidden_dim, metadata=meta, heads=num_heads)
        self.hgt2  = HGTConv(hidden_dim, hidden_dim, metadata=meta, heads=num_heads)
        self.skip  = Linear(hidden_dim, hidden_dim)
        self.bn    = nn.BatchNorm1d(hidden_dim)
        self.clf   = Linear(hidden_dim, out_dim)
    def forward(self, graph):
        xd = {'Patient': F.relu(self.p_lin(graph['Patient'].x)),
              'Gene':    F.relu(self.g_lin(graph['Gene'].x))}
        sp = self.skip(xd['Patient']); ei = graph.edge_index_dict
        xd = {k: F.dropout(F.relu(v), self.dr, training=self.training)
              for k,v in self.hgt1(xd, ei).items()}
        xd = self.hgt2(xd, ei)
        hp = self.bn(F.relu(xd['Patient'] + sp))
        return self.clf(F.dropout(hp, self.dr, training=self.training))


# ── 7. SGNN / ChebConv — V16 FIX: Gene->Patient aggregation added ─
# V15 bug: SGNN only used Patient.x + ChebConv on PP. Gene data ignored.
# V16 fix: GATv2Conv aggregates Gene->Patient FIRST, then ChebConv on PP.
class SGNNModel(nn.Module):
    def __init__(self, hidden_dim=32, out_dim=2, num_heads=4, dropout=0.2,
                 in_channels=PAT_FEAT_DIM, K=3, **_):
        super().__init__()
        self.dr    = dropout
        self.p_lin = nn.Linear(in_channels, hidden_dim)
        self.g_lin = Linear(-1, hidden_dim)          # Gene encoder — V16 FIX
        self.g2p   = GATv2Conv(hidden_dim, hidden_dim, heads=num_heads,
                               concat=False, add_self_loops=False)  # V16 FIX
        self.g2p_bn = nn.BatchNorm1d(hidden_dim)     # V16 FIX
        self.c1    = ChebConv(hidden_dim, hidden_dim, K=K)
        self.c2    = ChebConv(hidden_dim, hidden_dim, K=K)
        self.skip  = nn.Linear(hidden_dim, hidden_dim)
        self.bn1   = nn.BatchNorm1d(hidden_dim)
        self.bn2   = nn.BatchNorm1d(hidden_dim)
        self.clf   = nn.Linear(hidden_dim, out_dim)
    def forward(self, graph):
        xp    = graph['Patient'].x;  xg = graph['Gene'].x
        e_g2p = graph[('Gene','mutates','Patient')].edge_index
        pp_ei = graph[('Patient','cooccurs','Patient')].edge_index.to(xp.device)
        hp = F.relu(self.p_lin(xp))
        hg = F.relu(self.g_lin(xg))
        hp = self.g2p_bn(F.relu(hp + self.g2p((hg,hp), e_g2p)))  # V16 FIX
        s  = self.skip(hp)
        hp = self.bn1(F.relu(self.c1(hp, pp_ei)))
        hp = F.dropout(hp, self.dr, training=self.training)
        hp = self.bn2(F.relu(self.c2(hp, pp_ei) + s))
        return self.clf(F.dropout(hp, self.dr, training=self.training))


# ── Registry — V16: in_channels=PAT_FEAT_DIM=7 for fixed-dim models ─
MODEL_REGISTRY = [
    ('HeteroGATv2', HeteroGATv2,  {}),
    ('MOGAT',       MOGAT,        {}),
    ('HyperTMO',    HyperTMO,     {'in_channels': PAT_FEAT_DIM}),
    ('RGCN',        RGCNModel,    {'in_channels': PAT_FEAT_DIM, 'num_relations': 1}),  # V17 FIX 2: was NUM_GENES=20 but pp_et is always zeros — only 1 relation type is used
    ('VEGN',        VEGNModel,    {}),
    ('FastHGTConv', FastHGTModel, {'hgt_meta': _HGT_META_SF}),
    ('SGNN',        SGNNModel,    {'in_channels': PAT_FEAT_DIM}),
]
print(f'Registered {len(MODEL_REGISTRY)} models:', [n for n,_,_ in MODEL_REGISTRY])
print(f'V16 FIX: RGCN and SGNN now use Gene->Patient GATv2 aggregation before their PP-convolution.')
print(f'V16 FIX: All models receive 7-dim patient features (was 3 in V15).')


## 6b. Graph Audit — Verify All Topology Correctness

Checks all 4 bugs identified in V17 audit:
1. Scaler leakage: confirms scalers are fitted on train only
2. RGCN num_relations: confirms only relation 0 exists (now num_relations=1)
3. GCNConv self-loops: confirms no PP self-loops on `none` topology
4. PP edge key: confirms all 3 builders define the PP edge tensor


In [ ]:
print('='*65)
print('V17 GRAPH AUDIT — Verifying all 4 bug fixes')
print('='*65)

# ── Audit 1: Scaler data leakage ─────────────────────────────────
print('\n[1] Scaler data leakage')
assert _age_scaler    is not None, 'FAIL: _age_scaler not fitted'
assert _burden_scaler is not None, 'FAIL: _burden_scaler not fitted'
feat_train = _build_patient_features(train_df)
feat_test  = _build_patient_features(test_df)
feat_cgga  = _build_patient_features(cgga_df)
# Age column (index 2) — mean of test should NOT be ~0 (it would be if fit on test)
train_age_mean = feat_train[:, 2].mean()
test_age_mean  = feat_test[:,  2].mean()
cgga_age_mean  = feat_cgga[:, 2].mean()
print(f'  Age col mean — train: {train_age_mean:.4f}  test: {test_age_mean:.4f}  cgga: {cgga_age_mean:.4f}')
print(f'  (train≈0 by construction; test/CGGA ≠ 0 means scaler is train-fitted)')
if abs(train_age_mean) < 0.05 and abs(test_age_mean) > 0.01:
    print('  PASS: scalers fitted on train only (test/CGGA not centred at 0)')
else:
    print('  WARNING: check scaler fitting — expected train≈0, test/CGGA≠0')

# ── Audit 2: RGCN num_relations ──────────────────────────────────
print('\n[2] RGCN num_relations')
_rgcn = RGCNModel(hidden_dim=16, num_relations=1)
_rc1_weights = _rgcn.rc1.weight.shape
print(f'  RGCNConv weight shape: {list(_rc1_weights)}')
print(f'  Expected: [1, hidden, hidden] — 1 relation type')
assert _rc1_weights[0] == 1, f'FAIL: expected 1 relation, got {_rc1_weights[0]}'
print('  PASS: num_relations=1, matches pp_et=zeros usage')

# ── Audit 3: GCNConv self-loops on none topology ─────────────────
print('\n[3] GCNConv self-loops on none topology')
_g_none = build_graph(train_df.head(50), topology='none')
_pp_none = _g_none[('Patient','cooccurs','Patient')].edge_index
print(f'  PP edge count (none topology): {_pp_none.shape[1]}')
# Check GCNConv has add_self_loops=False
_hetgat = HeteroGATv2(hidden_dim=16)
_gcn_self_loops = _hetgat.pp_conv.add_self_loops
print(f'  HeteroGATv2.pp_conv.add_self_loops = {_gcn_self_loops}')
assert not _gcn_self_loops, 'FAIL: GCNConv still adds self-loops'
print('  PASS: no self-loops — none topology is truly PP-free')

# ── Audit 4: PP edge key present in all builders ──────────────────
print('\n[4] PP edge key present in all builders')
for topo in TOPOLOGY_NAMES:
    _g = build_graph(train_df.head(30), topology=topo)
    try:
        _ei = _g[('Patient','cooccurs','Patient')].edge_index
        print(f'  [{topo:8s}] PP edge_index shape: {list(_ei.shape)}  PASS')
    except Exception as e:
        print(f'  [{topo:8s}] FAIL: {e}')

# ── Audit 5: Model forward pass through all topologies ───────────
print('\n[5] All 7 models forward pass on all 3 topologies')
_test_params = {'hidden_dim':16,'num_heads':2,'dropout':0.1,'lr':1e-3,'weight_decay':1e-4}
for topo in TOPOLOGY_NAMES:
    _g = to_dev(build_graph(train_df.head(40), topology=topo))
    for mname, MCls, fkw in MODEL_REGISTRY:
        kw = {'hidden_dim':16,'out_dim':2,'num_heads':2,'dropout':0.1}
        kw.update(fkw)
        try:
            m = MCls(**kw).to(device)
            with torch.no_grad():
                out = m(_g)
            assert out.shape == (40, 2), f'bad output shape {out.shape}'
            print(f'  [{topo:8s}] {mname:14s} output={list(out.shape)}  PASS')
        except Exception as e:
            print(f'  [{topo:8s}] {mname:14s} FAIL: {e}')

print('\n' + '='*65)
print('AUDIT COMPLETE')
print('='*65)


## 7. Focal Loss + Softened Class Weights

**Fix**: penalty changed from `ratio` (≈1.38) to `sqrt(ratio)` (≈1.18).  
The FocalLoss `gamma=2` already down-weights easy examples; stacking a high class penalty over-corrects and inflates Grade-1 recall. Softening to square-root keeps a mild correction without making the decision boundary too aggressive toward Grade-1.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, weight=None):
        super().__init__()
        self.a, self.g, self.w = alpha, gamma, weight
    def forward(self, inp, tgt):
        ce = F.cross_entropy(inp, tgt, weight=self.w, reduction='none')
        return (self.a * (1 - torch.exp(-ce)) ** self.g * ce).mean()

counts  = train_df['Grade'].value_counts()
# ── FIX: soften penalty from ratio → sqrt(ratio) ─────────────────
penalty = float(np.sqrt(counts.max() / counts.min()))   # was counts.max()/counts.min()
cw      = torch.tensor([1.0, penalty], dtype=torch.float).to(device)
criterion = FocalLoss(alpha=1, gamma=2, weight=cw)
print(f"Class weights: [1.0, {penalty:.4f}]  (sqrt-penalty; was {counts.max()/counts.min():.4f})")

## 8. Threshold Optimisation + Evaluation Helpers

**Fix**: `find_optimal_threshold()` searches `[0.30, 0.75]` for the threshold maximising **G-mean = √(recall₀ × recall₁)**. G-mean is maximised when both recalls are balanced, so Grade-1 recall can no longer climb arbitrarily high at Grade-0's expense.

**AUC fix**: `compute_metrics()` uses explicit `try/except` around `roc_auc_score` and returns `nan` for degenerate folds rather than crashing.

In [ ]:
# ── FIX: G-mean threshold search ─────────────────────────────────
def find_optimal_threshold(probs, labels, low=0.30, high=0.75, step=0.01):
    best_th, best_gm = 0.5, -1.0
    for th in np.arange(low, high + step*0.5, step):
        preds = (probs >= th).astype(int)
        r1 = recall_score(labels, preds, pos_label=1, zero_division=0)
        r0 = recall_score(labels, preds, pos_label=0, zero_division=0)
        gm = np.sqrt(r0 * r1)
        if gm > best_gm: best_gm = gm; best_th = float(th)
    return best_th


def evaluate_model(model, graph, threshold=0.5):
    model.eval()
    with torch.no_grad():
        logits = model(graph)
        probs  = F.softmax(logits,1)[:,1].cpu().numpy()
        labels = graph['Patient'].y.cpu().numpy()
    preds = (probs >= threshold).astype(int)
    return preds, probs, labels


def compute_metrics(preds, probs, labels):
    try:    auc = roc_auc_score(labels, probs)
    except: auc = float('nan')
    return {
        'auc':      auc,
        'accuracy': accuracy_score(labels, preds),
        'precision':precision_score(labels, preds, zero_division=0),
        'recall':   recall_score(labels, preds, zero_division=0),
        'recall_0': recall_score(labels, preds, pos_label=0, zero_division=0),
        'f1':       f1_score(labels, preds, zero_division=0),
    }


# V17: storage containers — key is (model, pipeline, topology)
all_results    = []   # list of dicts; each has 'Graph_Topology' field
cv_results     = []   # per-fold CV results
all_models     = {}   # {(model_name, pipeline, topology): trained model}
all_studies    = {}   # {(model_name, pipeline, topology): optuna study}
all_params     = {}   # {(model_name, pipeline, topology): best_params}
all_thresholds = {}   # {(model_name, pipeline, topology): threshold}
PIPELINES      = ['No Balancing', 'SMOTE', 'CTGAN']
print('V17 storage containers initialised — key: (model, pipeline, topology)')


## 9. Unified Training Function

In [ ]:
def train_and_evaluate(train_graph, val_graph, params, ModelClass,
                        fixed_kw=None, max_epochs=MAX_EPOCHS, patience=PATIENCE,
                        seed=42):
    """Train model; return (best_val_auc, best_state, history)."""
    set_seed(seed)   # reset RNG before every model init+train for reproducibility
    kw = {'hidden_dim': params['hidden_dim'], 'out_dim': 2,
          'num_heads':  params['num_heads'],  'dropout': params['dropout']}
    if fixed_kw: kw.update(fixed_kw)

    model = ModelClass(**kw).to(device)
    try:
        with torch.no_grad(): _ = model(train_graph)
    except Exception: pass

    opt = torch.optim.AdamW(model.parameters(),
                              lr=params['lr'], weight_decay=params['weight_decay'])
    sch = CosineAnnealingLR(opt, T_max=max_epochs, eta_min=params['lr'] * 0.05)

    best_auc, ctr, best_state, history = 0.0, 0, None, []
    for _ in range(max_epochs):
        model.train(); opt.zero_grad()
        loss = criterion(model(train_graph), train_graph['Patient'].y)
        loss.backward(); opt.step(); sch.step()

        model.eval()
        with torch.no_grad():
            vp  = F.softmax(model(val_graph), 1)[:, 1].cpu().numpy()
            vl  = val_graph['Patient'].y.cpu().numpy()
        try:
            auc = roc_auc_score(vl, vp)
        except ValueError:
            auc = 0.0
        history.append(auc)

        if auc > best_auc:
            best_auc, ctr = auc, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            ctr += 1
            if ctr >= patience: break

    return best_auc, best_state, history

## 10. Optuna HPO (warm-up on train_df / val_df)

In [ ]:
def run_optuna(train_df_raw, val_graph_topo, ModelClass, fixed_kw=None,
               n_trials=N_TRIALS, label='',
               augment_fn=None, topology='ba'):
    """Bayesian HPO with topology-aware graph building.

    topology : 'none' | 'jaccard' | 'ba'
      - none   : no topology params added to search space
      - jaccard: top_k in {3..10} added to search space
      - ba     : ba_m  in {1..5}  added to search space
    """
    def objective(trial):
        set_seed(42)
        params = {
            'hidden_dim':   trial.suggest_categorical('hidden_dim', [16,32,64,128]),
            'num_heads':    trial.suggest_categorical('num_heads',  [2,4,8]),
            'dropout':      trial.suggest_float('dropout', 0.05, 0.30, step=0.05),
            'lr':           trial.suggest_float('lr', 1e-4, 1e-2, log=True),
            'weight_decay': trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True),
        }
        # Topology-specific hyperparameters
        if topology == 'ba':
            params['ba_m']  = trial.suggest_int('ba_m', 1, 5)
        elif topology == 'jaccard':
            params['top_k'] = trial.suggest_int('top_k', 3, 10)
        # topology='none': no extra params

        df_t = augment_fn(train_df_raw.copy()) if augment_fn else train_df_raw
        train_g = to_dev(build_graph(df_t, topology=topology,
                                     ba_m=params.get('ba_m',2),
                                     top_k=params.get('top_k',5)))
        auc, _, _ = train_and_evaluate(train_g, val_graph_topo, params, ModelClass, fixed_kw)
        clear_pp_cache()
        return auc

    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    bp = study.best_params
    print(f'  [{label}] topology={topology}  Best AUC={study.best_value:.4f}  params={bp}')

    # Rebuild best train graph for final state
    df_best  = augment_fn(train_df_raw.copy()) if augment_fn else train_df_raw
    best_tr_g = to_dev(build_graph(df_best, topology=topology,
                                    ba_m=bp.get('ba_m',2),
                                    top_k=bp.get('top_k',5)))
    clear_pp_cache()
    _, best_state, _ = train_and_evaluate(best_tr_g, val_graph_topo, bp, ModelClass, fixed_kw)
    return bp, best_state, study


## 11. 5-Fold Stratified Cross-Validation

Uses the **fixed best params** found by Optuna.  Each fold: augment training portion → build graph → train → find G-mean threshold on val → evaluate.  
Reports mean ± std across folds for AUC, Accuracy, Precision, Recall (Grade-1 & Grade-0), F1, Threshold.

In [ ]:
def apply_smote(fold_train_df):
    feat_cols = gene_columns + ['Gender','Race','Age_at_diagnosis']
    cat_idx   = [i for i,c in enumerate(feat_cols)
                  if c in gene_columns or c in ['Gender','Race']]
    sm = SMOTENC(categorical_features=cat_idx, random_state=42, k_neighbors=3)
    Xr, yr = sm.fit_resample(fold_train_df[feat_cols], fold_train_df['Grade'])
    df2 = pd.DataFrame(Xr, columns=feat_cols); df2['Grade'] = yr
    for c in gene_columns: df2[c] = df2[c].round().astype(int)
    return df2


def apply_ctgan(fold_train_df):
    meta = SingleTableMetadata()
    meta.detect_from_dataframe(fold_train_df)
    for col in categorical_columns + gene_columns:
        meta.update_column(column_name=col, sdtype='categorical')
    vc    = fold_train_df['Grade'].value_counts()
    n_gen = int(vc.max() - vc.min())
    if n_gen <= 0: return fold_train_df
    syn = CTGANSynthesizer(meta, epochs=100, batch_size=50, verbose=False, cuda=True, pac=10)
    torch.manual_seed(42); np.random.seed(42)
    syn.fit(fold_train_df)
    cond  = Condition(num_rows=n_gen, column_values={'Grade': int(vc.idxmin())})
    extra = syn.sample_from_conditions(conditions=[cond])
    return pd.concat([fold_train_df, extra], ignore_index=True)


def run_5fold_cv(train_val_df, best_params, ModelClass, fixed_kw,
                  pipeline_name, model_name,
                  augment_fn=None, topology='ba'):
    """5-Fold CV — V17: topology-aware graph building per fold."""
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    fold_records = []
    ba_m  = best_params.get('ba_m',  2)
    top_k = best_params.get('top_k', 5)

    for fold, (tr_idx, vl_idx) in enumerate(
            skf.split(train_val_df, train_val_df['Grade']), start=1):
        fold_tr = train_val_df.iloc[tr_idx].copy().reset_index(drop=True)
        fold_vl = train_val_df.iloc[vl_idx].copy().reset_index(drop=True)
        set_seed(42)
        if augment_fn is not None: fold_tr = augment_fn(fold_tr)
        clear_pp_cache()
        tr_g = to_dev(build_graph(fold_tr, topology=topology, ba_m=ba_m, top_k=top_k))
        vl_g = to_dev(build_graph(fold_vl, topology=topology, ba_m=ba_m, top_k=top_k))
        _, state, _ = train_and_evaluate(tr_g, vl_g, best_params, ModelClass, fixed_kw)
        kw = {'hidden_dim':best_params['hidden_dim'],'out_dim':2,
              'num_heads':best_params['num_heads'],'dropout':best_params['dropout']}
        kw.update(fixed_kw)
        fold_model = ModelClass(**kw).to(device)
        try:
            with torch.no_grad(): _ = fold_model(tr_g)
        except: pass
        fold_model.load_state_dict(state)
        _, probs_v, labels_v = evaluate_model(fold_model, vl_g, threshold=0.5)
        th = find_optimal_threshold(probs_v, labels_v)
        preds_v = (probs_v >= th).astype(int)
        m = compute_metrics(preds_v, probs_v, labels_v)
        m.update({'fold':fold,'threshold':th,'Model':model_name,
                  'Pipeline':pipeline_name,'Graph_Topology':topology})
        fold_records.append(m)
        print(f'    Fold {fold}/5 | AUC={m["auc"]:.4f} '
              f'R1={m["recall"]:.3f} R0={m["recall_0"]:.3f} '
              f'F1={m["f1"]:.4f} th={th:.2f}')
    clear_pp_cache()
    return pd.DataFrame(fold_records)


## 12. Pipeline Runner (shared by all 3 pipelines)

In [ ]:
def run_pipeline(pipeline_name, train_df_raw, augment_fn=None):
    """V17: runs all 3 topologies for every model in this pipeline.

    For each (model, topology):
      1. Optuna HPO — topology-specific graph built per trial
      2. 5-Fold CV  — topology-specific graph built per fold
      3. Final model on full train_val
      4. Evaluate on topology-matched test + CGGA graphs
    """
    print(f'\n{"="*75}')
    print(f'PIPELINE: {pipeline_name}  |  Topologies: {TOPOLOGY_NAMES}')
    print(f'{"="*75}')

    for topology in TOPOLOGY_NAMES:
        print(f'\n  ── Topology: {topology.upper()} ──────────────────────────────')
        val_g_topo  = VAL_GRAPHS[topology]
        test_g_topo = TEST_GRAPHS[topology]
        cgga_g_topo = CGGA_GRAPHS[topology]

        for mname, MCls, fkw in MODEL_REGISTRY:
            key = (mname, pipeline_name, topology)
            print(f'\n    [{topology}] {mname}')

            # Step 1: Optuna HPO
            bp, _, study = run_optuna(
                train_df_raw, val_g_topo, MCls, fkw,
                augment_fn=augment_fn, topology=topology,
                label=f'{mname}/{pipeline_name}/{topology}')
            all_studies[key] = study
            all_params[key]  = bp
            ba_m  = bp.get('ba_m',  2)
            top_k = bp.get('top_k', 5)

            # Step 2: 5-Fold CV
            cv_df = run_5fold_cv(train_val_df, bp, MCls, fkw,
                                  pipeline_name, mname,
                                  augment_fn=augment_fn, topology=topology)
            cv_results.append(cv_df)
            for col in ['auc','recall','recall_0','f1','threshold']:
                if col in cv_df.columns:
                    print(f'      {col:12s}: {cv_df[col].mean():.4f} +/- {cv_df[col].std():.4f}')

            # Step 3: Final model on full train_val
            aug_df = augment_fn(train_val_df.copy()) if augment_fn else train_val_df
            clear_pp_cache()
            full_tr_g = to_dev(build_graph(aug_df, topology=topology,
                                            ba_m=ba_m, top_k=top_k))
            _, final_state, _ = train_and_evaluate(full_tr_g, val_g_topo, bp, MCls, fkw)
            kw = {'hidden_dim':bp['hidden_dim'],'out_dim':2,
                  'num_heads':bp['num_heads'],'dropout':bp['dropout']}
            kw.update(fkw)
            final_model = MCls(**kw).to(device)
            try:
                with torch.no_grad(): _ = final_model(full_tr_g)
            except: pass
            final_model.load_state_dict(final_state)

            th = float(cv_df['threshold'].mean())
            all_thresholds[key] = th
            all_models[key]     = final_model
            print(f'      Threshold={th:.3f}  ba_m={ba_m}  top_k={top_k}')

            # Step 4: Evaluate on topology-matched test + CGGA
            pdt, pbt, lbt = evaluate_model(final_model, test_g_topo, threshold=th)
            pdc, pbc, lbc = evaluate_model(final_model, cgga_g_topo, threshold=th)
            mt  = compute_metrics(pdt, pbt, lbt)
            mc_ = compute_metrics(pdc, pbc, lbc)
            print(f'      TCGA-Test AUC={mt["auc"]:.4f} R1={mt["recall"]:.3f} R0={mt["recall_0"]:.3f} F1={mt["f1"]:.4f}')
            print(f'      CGGA      AUC={mc_["auc"]:.4f} R1={mc_["recall"]:.3f} R0={mc_["recall_0"]:.3f} F1={mc_["f1"]:.4f}')

            for ds, m, p, l in [('TCGA Test',mt,pbt,lbt),('CGGA',mc_,pbc,lbc)]:
                rec = {'Model':mname,'Pipeline':pipeline_name,
                       'Graph_Topology':topology,'Dataset':ds,
                       'threshold':th,'probs':p,'labels':l,
                       'ba_m':ba_m,'top_k':top_k}
                rec.update(m); all_results.append(rec)
            clear_pp_cache()

    print(f'\n[{pipeline_name}] All 3 topologies complete.')


## 13. Pipeline A — No Balancing

In [ ]:
run_pipeline('No Balancing', train_df_raw=train_df, augment_fn=None)


## 14. Pipeline B — SMOTE

In [ ]:
run_pipeline('SMOTE', train_df_raw=train_df, augment_fn=apply_smote)


## 15. Pipeline C — CTGAN

In [ ]:
run_pipeline('CTGAN', train_df_raw=train_df, augment_fn=apply_ctgan)
print('\nPipelines A / B / C complete — all 3 topologies each.')


## 4c. ROS Augmentation Function

In [ ]:
from imblearn.over_sampling import RandomOverSampler


def apply_ros(fold_train_df, seed=42):
    """
    Random Over-Sampling: duplicates minority-class rows at random until balanced.

    - Creates NO synthetic samples (unlike SMOTE) -> safe for binary gene columns
    - Preserves true mutation co-occurrence patterns
    - API matches apply_smote() -> plug-in compatible with run_pipeline()
    """
    feat_cols = gene_columns + ["Gender", "Race", "Age_at_diagnosis"]
    ros = RandomOverSampler(sampling_strategy="auto", random_state=seed)
    Xr, yr = ros.fit_resample(fold_train_df[feat_cols], fold_train_df["Grade"])
    df2 = pd.DataFrame(Xr, columns=feat_cols)
    df2["Grade"] = yr
    for c in gene_columns: df2[c] = df2[c].astype(int)
    df2["Gender"] = df2["Gender"].astype(int)
    df2["Race"]   = df2["Race"].astype(int)
    print(f"    [ROS] Grade dist after balancing: {dict(pd.Series(yr).value_counts())}")
    return df2


print(f"Original train Grade dist: {dict(train_df['Grade'].value_counts())}")
_ = apply_ros(train_df.copy())
print("apply_ros() defined and verified.")


## Pipeline D — ROS (Random Over-Sampling)

In [ ]:
from imblearn.over_sampling import RandomOverSampler

def apply_ros(fold_train_df, seed=42):
    feat_cols = gene_columns + ['Gender','Race','Age_at_diagnosis']
    ros = RandomOverSampler(sampling_strategy='auto', random_state=seed)
    Xr, yr = ros.fit_resample(fold_train_df[feat_cols], fold_train_df['Grade'])
    df2 = pd.DataFrame(Xr, columns=feat_cols); df2['Grade'] = yr
    for c in gene_columns: df2[c] = df2[c].astype(int)
    df2['Gender'] = df2['Gender'].astype(int)
    df2['Race']   = df2['Race'].astype(int)
    return df2

print('\n' + '='*75)
print('PIPELINE D: ROS — all 3 topologies')
print('='*75)
PIPELINES.append('ROS')
run_pipeline('ROS', train_df_raw=train_df, augment_fn=apply_ros)
print('\nROS pipeline complete.')


## Pipeline E — AUC-Weighted Ensemble (per pipeline × per topology)

Ensemble is built separately for each pipeline × topology combination.
Stored as `Pipeline='Ensemble'`, `Graph_Topology=topology`.


In [ ]:
def ensemble_predict(model_keys, graph, weights=None):
    probs_list = []
    for key in model_keys:
        m = all_models.get(key)
        if m is None: continue
        _, probs, _ = evaluate_model(m, graph, threshold=0.5)
        probs_list.append(probs)
    if not probs_list: return np.zeros(graph['Patient'].x.shape[0])
    w = np.array(weights[:len(probs_list)] if weights else [1.0]*len(probs_list), dtype=float)
    w = w / w.sum()
    return sum(p*wi for p,wi in zip(probs_list, w))


print('\n' + '='*75)
print('PIPELINE E: AUC-Weighted Ensemble (per pipeline × topology)')
print('='*75)
PIPELINES.append('Ensemble')
model_names_all = [n for n,_,_ in MODEL_REGISTRY]

for ens_pipe in ['No Balancing','SMOTE','CTGAN','ROS']:
    for topology in TOPOLOGY_NAMES:
        pipe_keys = [(mn, ens_pipe, topology) for mn in model_names_all
                      if (mn, ens_pipe, topology) in all_models]
        if not pipe_keys: continue

        auc_weights = []
        for key in pipe_keys:
            row = next((r for r in all_results
                        if r['Model']==key[0] and r['Pipeline']==key[1]
                        and r['Graph_Topology']==key[2]
                        and r['Dataset']=='TCGA Test'), None)
            w_val = float(row['auc']) if row and row['auc']==row['auc'] else 0.5
            auc_weights.append(w_val)

        test_g = TEST_GRAPHS[topology]; cgga_g = CGGA_GRAPHS[topology]
        ep_test = ensemble_predict(pipe_keys, test_g, weights=auc_weights)
        el_test = test_g['Patient'].y.cpu().numpy()
        ens_th  = find_optimal_threshold(ep_test, el_test)
        mt_ens  = compute_metrics((ep_test>=ens_th).astype(int), ep_test, el_test)
        ep_cgga = ensemble_predict(pipe_keys, cgga_g, weights=auc_weights)
        el_cgga = cgga_g['Patient'].y.cpu().numpy()
        mc_ens  = compute_metrics((ep_cgga>=ens_th).astype(int), ep_cgga, el_cgga)

        print(f'  Ensemble({ens_pipe}/{topology}): '
              f'TCGA AUC={mt_ens["auc"]:.4f}  CGGA AUC={mc_ens["auc"]:.4f}')
        for ds,m,p,l in [('TCGA Test',mt_ens,ep_test,el_test),
                          ('CGGA',mc_ens,ep_cgga,el_cgga)]:
            rec = {'Model':f'Ensemble({ens_pipe})','Pipeline':'Ensemble',
                   'Graph_Topology':topology,'Dataset':ds,
                   'threshold':ens_th,'probs':p,'labels':l,'ba_m':'N/A','top_k':'N/A'}
            rec.update(m); all_results.append(rec)

print(f'\nEnsemble complete. Total results: {len(all_results)}')


## 16. 5-Fold CV Summary Table

In [ ]:
cv_all = pd.concat(cv_results, ignore_index=True)

cv_compact = cv_all.groupby(['Model','Pipeline']).apply(
    lambda g: pd.Series({
        'AUC':       f"{g['auc'].mean():.4f} ± {g['auc'].std():.4f}",
        'Recall-1':  f"{g['recall'].mean():.4f} ± {g['recall'].std():.4f}",
        'Recall-0':  f"{g['recall_0'].mean():.4f} ± {g['recall_0'].std():.4f}",
        'F1':        f"{g['f1'].mean():.4f} ± {g['f1'].std():.4f}",
        'Threshold': f"{g['threshold'].mean():.3f} ± {g['threshold'].std():.3f}",
    })
).reset_index()

print("\n5-FOLD CV SUMMARY (mean ± std across 5 folds)")
print("="*90)
print(cv_compact.to_string(index=False))

## 17. Final Results Table (TCGA Test + CGGA)

**AUC fix**: columns built explicitly by name — no brittle positional assignment.

In [ ]:
_raw = []
for r in all_results:
    _raw.append({
        'Model':          r['Model'],
        'Pipeline':       r['Pipeline'],
        'Graph_Topology': r.get('Graph_Topology','ba'),
        'Dataset':        r['Dataset'],
        'Threshold':      round(r['threshold'],3),
        'AUC':            round(r['auc'],4),
        'Accuracy':       round(r['accuracy'],4),
        'Precision':      round(r['precision'],4),
        'Recall_1':       round(r['recall'],4),
        'Recall_0':       round(r['recall_0'],4),
        'F1':             round(r['f1'],4),
    })

results_df = (pd.DataFrame(_raw)
              .drop_duplicates(subset=['Model','Pipeline','Graph_Topology','Dataset'], keep='last')
              .reset_index(drop=True))

print('\n' + '='*130)
print('FINAL RESULTS — 7 MODELS × 4 PIPELINES × 3 TOPOLOGIES × 2 DATASETS')
print('='*130)
print(results_df.to_string(index=False))

# Best per (model, topology) across pipelines
print('\n' + '='*130)
print('TOPOLOGY COMPARISON — Best pipeline per (Model, Topology) — TCGA Test')
print('='*130)
sub = results_df[results_df.Dataset=='TCGA Test']
best_t = (sub.sort_values('AUC',ascending=False)
             .drop_duplicates(['Model','Graph_Topology'])
             [['Model','Graph_Topology','Pipeline','AUC','Recall_1','Recall_0','F1']])
print(best_t.to_string(index=False))

# Best topology per model (which topology wins most often)
print('\nTopology win counts on TCGA Test (best AUC per model):')
top_per_model = (sub.sort_values('AUC',ascending=False)
                    .drop_duplicates('Model')[['Model','Graph_Topology','Pipeline','AUC']])
print(top_per_model.to_string(index=False))
print('\nTopology wins:', dict(top_per_model['Graph_Topology'].value_counts()))

# Top-10 overall
print('\nTop-10 overall by AUC on TCGA Test:')
print(sub.sort_values('AUC',ascending=False).head(10)
      [['Model','Pipeline','Graph_Topology','AUC','Recall_1','Recall_0','F1']].to_string(index=False))

results_df.to_csv('V17_results_final.csv', index=False)
print('\nExported: V17_results_final.csv')
if cv_results:
    cv_all_df = pd.concat(cv_results, ignore_index=True)
    cv_all_df.to_csv('V17_cv_folds.csv', index=False)
    print('Exported: V17_cv_folds.csv')


## 18. AUC Heatmaps

In [ ]:
# ── AUC Heatmap: Model × Pipeline, one subplot per topology per dataset ──
fig, axes = plt.subplots(len(TOPOLOGY_NAMES), 2,
                          figsize=(16, 5*len(TOPOLOGY_NAMES)))
for ti, topo in enumerate(TOPOLOGY_NAMES):
    for di, ds in enumerate(['TCGA Test','CGGA']):
        ax = axes[ti, di]
        sub = results_df[(results_df.Dataset==ds) & (results_df.Graph_Topology==topo)]
        if sub.empty: ax.set_visible(False); continue
        pivot = sub.pivot_table(index='Model', columns='Pipeline',
                                values='AUC', aggfunc='mean')
        sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGnBu',
                    linewidths=0.5, ax=ax, vmin=0.5, vmax=1.0,
                    cbar_kws={'label':'AUC'})
        ax.set_title(f'AUC — {ds} | Topology: {topo}', fontsize=11)
        ax.set_xlabel('Pipeline'); ax.set_ylabel('Model')
plt.suptitle('AUC Heatmaps: All Models × Pipelines × Topologies', fontsize=13)
plt.tight_layout()
plt.savefig('V17_auc_heatmap_by_topology.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Topology comparison: mean AUC per topology per model (TCGA Test) ──
fig, ax = plt.subplots(figsize=(14, 5))
sub_test = results_df[results_df.Dataset=='TCGA Test']
topo_model = (sub_test.groupby(['Model','Graph_Topology'])['AUC']
              .mean().unstack('Graph_Topology').reindex(columns=TOPOLOGY_NAMES))
topo_model.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white', width=0.7)
ax.set_title('Mean AUC per Model × Topology — TCGA Test', fontsize=13)
ax.set_xlabel('Model'); ax.set_ylabel('Mean AUC (across pipelines)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.legend(title='PP Topology', fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.axhline(topo_model.values.mean(), color='red', ls='--', lw=1, alpha=0.5,
           label='Grand mean')
plt.tight_layout()
plt.savefig('V17_topology_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Print topology ranking table ──
print('\nMean AUC per Topology (TCGA Test, averaged over all models & pipelines):')
print(sub_test.groupby('Graph_Topology')['AUC'].agg(['mean','std','max'])
      .round(4).sort_values('mean',ascending=False).to_string())
print('\nMean AUC per Topology (CGGA):')
print(results_df[results_df.Dataset=='CGGA'].groupby('Graph_Topology')['AUC']
      .agg(['mean','std','max']).round(4).sort_values('mean',ascending=False).to_string())


## 19. Recall Heatmaps — Grade-1 and Grade-0 (TCGA Test)

In [ ]:
# Recall heatmaps — one figure per topology
for topo in TOPOLOGY_NAMES:
    sub_t = results_df[(results_df.Dataset=='TCGA Test') &
                        (results_df.Graph_Topology==topo)]
    if sub_t.empty: continue
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, col, title in [
            (axes[0],'Recall_1','Grade-1 Recall (sensitivity)'),
            (axes[1],'Recall_0','Grade-0 Recall (specificity)')]:
        pivot = sub_t.pivot_table(index='Model',columns='Pipeline',
                                   values=col,aggfunc='mean')
        sns.heatmap(pivot,annot=True,fmt='.4f',cmap='RdYlGn',
                    linewidths=0.5,ax=ax,vmin=0.4,vmax=1.0,
                    cbar_kws={'label':col})
        ax.set_title(f'{title} | {topo}'); ax.set_xlabel('Pipeline'); ax.set_ylabel('Model')
    plt.suptitle(f'Recall Heatmap — TCGA Test | Topology: {topo}', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'V17_recall_{topo}.png', dpi=150, bbox_inches='tight')
    plt.show()


## 20. 5-Fold CV Box Plots

In [ ]:
metrics_cv = ['auc', 'recall', 'recall_0', 'f1']
labels_cv  = ['AUC', 'Recall Grade-1', 'Recall Grade-0', 'F1']
model_names = [n for n, _, _ in MODEL_REGISTRY]

for pipe in PIPELINES:
    sub = cv_all[cv_all.Pipeline == pipe]
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    for ax, met, lab in zip(axes, metrics_cv, labels_cv):
        data = [sub[sub.Model == m][met].values for m in model_names]
        bp = ax.boxplot(data, patch_artist=True, notch=False)
        colors = plt.cm.tab10(np.linspace(0, 0.9, len(model_names)))
        for patch, col in zip(bp['boxes'], colors):
            patch.set_facecolor(col)
        ax.set_xticks(range(1, len(model_names)+1))
        ax.set_xticklabels(model_names, rotation=45, ha='right', fontsize=8)
        ax.set_ylabel(lab); ax.set_title(lab); ax.grid(axis='y', alpha=0.3)
        ax.axhline(0.5, color='red', ls='--', alpha=0.4, lw=0.8)
    plt.suptitle(f'5-Fold CV Distribution — {pipe}', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'V15_cv_boxplot_{pipe.replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 21. ROC Curve Grid

In [ ]:
results_full = pd.DataFrame(all_results)
topo_colors  = {'none':'steelblue', 'jaccard':'forestgreen', 'ba':'tomato'}
topo_styles  = {'none':'-', 'jaccard':'--', 'ba':':'}
model_names  = [n for n,_,_ in MODEL_REGISTRY]

# ROC grid: rows=models, cols=datasets; lines=topology (best pipeline per topology)
fig, axes = plt.subplots(len(model_names), 2,
                          figsize=(13, 3.5*len(model_names)))
for row, mname in enumerate(model_names):
    for col, ds in enumerate(['TCGA Test','CGGA']):
        ax = axes[row, col]
        for topo in TOPOLOGY_NAMES:
            sub = results_full[(results_full.Model==mname) &
                                (results_full.Graph_Topology==topo) &
                                (results_full.Dataset==ds)]
            if sub.empty: continue
            # Best pipeline for this topology
            best = sub.loc[sub['auc'].idxmax()]
            fpr, tpr, _ = roc_curve(best['labels'], best['probs'])
            ax.plot(fpr, tpr, color=topo_colors[topo],
                    ls=topo_styles[topo], lw=1.8,
                    label=f'{topo} (AUC={best["auc"]:.3f})')
        ax.plot([0,1],[0,1],'k--',alpha=0.3)
        ax.set_title(f'{mname} — {ds}', fontsize=8)
        ax.set_xlabel('FPR',fontsize=7); ax.set_ylabel('TPR',fontsize=7)
        ax.legend(fontsize=6,title='Topology'); ax.grid(alpha=0.3)
plt.suptitle('ROC Curves — Best Pipeline per Topology per Model', fontsize=13, y=1.002)
plt.tight_layout()
plt.savefig('V17_roc_by_topology.png', dpi=150, bbox_inches='tight')
plt.show()


## 22. Confusion Matrices

In [ ]:
for ds in ['TCGA Test','CGGA']:
    cmap = 'Blues' if ds=='TCGA Test' else 'Oranges'
    fig, axes = plt.subplots(1, len(TOPOLOGY_NAMES),
                              figsize=(6*len(TOPOLOGY_NAMES), 4))
    for ti, topo in enumerate(TOPOLOGY_NAMES):
        sub = results_full[(results_full.Dataset==ds) &
                            (results_full.Graph_Topology==topo)]
        if sub.empty: axes[ti].set_visible(False); continue
        best = sub.loc[sub['auc'].idxmax()]
        preds = (best['probs'] >= best['threshold']).astype(int)
        cm    = confusion_matrix(best['labels'], preds)
        ConfusionMatrixDisplay(cm,display_labels=['Grade 0','Grade 1']).plot(
            ax=axes[ti], cmap=cmap, colorbar=False)
        axes[ti].set_title(f'Topology: {topo}\n{best["Model"]} / {best["Pipeline"]}\n'
                           f'AUC={best["auc"]:.3f}', fontsize=9)
    plt.suptitle(f'Confusion Matrices — {ds} (best per topology)', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'V17_cm_{ds.replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()


## 23. Permutation Feature Importance — Genes + Clinical Features

For every feature (20 gene mutations + 3 clinical: **Age_at_diagnosis, Gender, Race**),
the column is shuffled `n_repeats=10` times and the mean AUC drop is recorded.

- **Gene features** are permuted as binary mutation columns
- **Clinical features** are permuted as continuous/categorical columns in the same way
- The graph is rebuilt from scratch after each permutation so the message-passing
  structure reflects the shuffled values
- Larger AUC drop → feature is more important for classification

In [ ]:
CLINICAL_FEATURES = ['Age_at_diagnosis','Gender','Race']
ALL_FEATURES      = gene_columns + CLINICAL_FEATURES

def perm_importance_all(model, ref_df, topology='ba', threshold=0.5, n_repeats=10, seed=0):
    base_g = to_dev(build_graph(ref_df, topology=topology))
    model.eval()
    with torch.no_grad():
        bp = F.softmax(model(base_g),1)[:,1].cpu().numpy()
    try:    base_auc = roc_auc_score(ref_df['Grade'].values, bp)
    except: return {f:0.0 for f in ALL_FEATURES}, 0.0
    rng = np.random.default_rng(seed); out = {}
    for feat in ALL_FEATURES:
        drops = []
        for _ in range(n_repeats):
            df2 = ref_df.copy()
            df2[feat] = rng.permutation(df2[feat].values)
            g2 = to_dev(build_graph(df2, topology=topology))
            with torch.no_grad():
                pp = F.softmax(model(g2),1)[:,1].cpu().numpy()
            try:    drops.append(base_auc - roc_auc_score(ref_df['Grade'].values, pp))
            except: drops.append(0.0)
        out[feat] = float(np.mean(drops))
    clear_pp_cache()
    return out, base_auc

print('Computing permutation importance (one best model per topology)...')
imp_records = []
for topo in TOPOLOGY_NAMES:
    # Use best model × pipeline for this topology
    sub_t = results_df[(results_df.Dataset=='TCGA Test') &
                        (results_df.Graph_Topology==topo)]
    if sub_t.empty: continue
    best_row = sub_t.loc[sub_t['AUC'].idxmax()]
    mname, pipe = best_row['Model'], best_row['Pipeline']
    model = all_models.get((mname, pipe, topo))
    if model is None: continue
    th = all_thresholds.get((mname, pipe, topo), 0.5)
    print(f'  [{topo}] Best: {mname}/{pipe}  AUC={best_row["AUC"]:.4f}')
    imp, base = perm_importance_all(model, test_df, topology=topo, threshold=th)
    for feat, drop in imp.items():
        imp_records.append({'Model':mname,'Pipeline':pipe,'Graph_Topology':topo,
                            'Feature':feat,'AUC_Drop':drop,'Base_AUC':base,
                            'Feature_Type':'Clinical' if feat in CLINICAL_FEATURES else 'Gene'})

imp_df = pd.DataFrame(imp_records)
imp_df.to_csv('V17_feature_importance.csv', index=False)
print(f'Done. Saved V17_feature_importance.csv ({len(imp_df)} rows)')


## 24. Feature Importance Plots — Genes + Clinical

In [ ]:
# ── Aggregate: mean AUC drop per feature across all models & pipelines ──
mean_imp = (imp_df.groupby(['Feature','Feature_Type'])['AUC_Drop']
            .mean()
            .reset_index()
            .sort_values('AUC_Drop', ascending=False)
            .reset_index(drop=True))

# Colour: orange for clinical, steelblue for gene
bar_colors = ['#E87722' if t == 'Clinical' else '#4C8AC4'
              for t in mean_imp['Feature_Type']]

# ─────────────────────────────────────────────────────────────────
# PLOT A: Combined bar chart — all 23 features, colour-coded by type
# ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
bars = ax.bar(mean_imp['Feature'], mean_imp['AUC_Drop'], color=bar_colors, edgecolor='white')
ax.set_xlabel('Feature'); ax.set_ylabel('Mean AUC Drop')
ax.set_title('Permutation Feature Importance — Genes (blue) + Clinical (orange)', fontsize=13)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.grid(axis='y', alpha=0.3)
for b, v in zip(bars, mean_imp['AUC_Drop']):
    ax.text(b.get_x() + b.get_width()/2, max(v + 0.0003, 0.0005),
            f'{v:.4f}', ha='center', va='bottom', fontsize=6.5)

# Custom legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#4C8AC4', label='Gene mutation'),
                    Patch(color='#E87722', label='Clinical feature')],
           fontsize=9, loc='upper right')
plt.tight_layout()
plt.savefig('V15_feat_importance_combined.png', dpi=150, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────
# PLOT B: Side-by-side — Gene panel | Clinical panel
# ─────────────────────────────────────────────────────────────────
gene_imp = mean_imp[mean_imp.Feature_Type == 'Gene'].copy()
clin_imp = mean_imp[mean_imp.Feature_Type == 'Clinical'].copy()

fig, axes = plt.subplots(1, 2, figsize=(20, 5),
                          gridspec_kw={'width_ratios': [len(gene_imp), len(clin_imp)]})

# Gene panel
gcols = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(gene_imp)))
b1 = axes[0].bar(gene_imp['Feature'], gene_imp['AUC_Drop'], color=gcols, edgecolor='white')
axes[0].set_title('Gene Mutation Importance', fontsize=12)
axes[0].set_xlabel('Gene'); axes[0].set_ylabel('Mean AUC Drop')
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
axes[0].grid(axis='y', alpha=0.3)
for b, v in zip(b1, gene_imp['AUC_Drop']):
    axes[0].text(b.get_x() + b.get_width()/2, max(v + 0.0003, 0.0005),
                  f'{v:.4f}', ha='center', va='bottom', fontsize=7)

# Clinical panel
ccols = plt.cm.Oranges(np.linspace(0.45, 0.85, len(clin_imp)))
b2 = axes[1].bar(clin_imp['Feature'], clin_imp['AUC_Drop'], color=ccols, edgecolor='white',
                  width=0.5)
axes[1].set_title('Clinical Feature Importance', fontsize=12)
axes[1].set_xlabel('Clinical Feature'); axes[1].set_ylabel('Mean AUC Drop')
axes[1].tick_params(axis='x', rotation=0, labelsize=10)
axes[1].grid(axis='y', alpha=0.3)
for b, v in zip(b2, clin_imp['AUC_Drop']):
    axes[1].text(b.get_x() + b.get_width()/2, max(v + 0.0003, 0.0005),
                  f'{v:.4f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Feature Importance by Category', fontsize=13)
plt.tight_layout()
plt.savefig('V15_feat_importance_sidebyside.png', dpi=150, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────
# PLOT C: Heatmap — all 23 features × 7 models (averaged over pipelines)
# ─────────────────────────────────────────────────────────────────
# Column order: clinical first, then genes sorted by mean importance
col_order = (list(clin_imp['Feature']) +
             list(gene_imp.sort_values('AUC_Drop', ascending=False)['Feature']))

heat = (imp_df.groupby(['Model','Feature'])['AUC_Drop']
        .mean()
        .unstack('Feature')
        .reindex(columns=col_order))

fig, ax = plt.subplots(figsize=(22, 5))
sns.heatmap(heat, annot=True, fmt='.4f', cmap='YlOrRd', linewidths=0.35,
            ax=ax, cbar_kws={'label': 'Mean AUC Drop'})

# Vertical separator between clinical and gene columns
ax.axvline(x=len(CLINICAL_FEATURES), color='navy', lw=2.5, ls='--')

ax.set_title('Permutation Importance Heatmap — All Features × All Models\n'
             '(dashed line separates Clinical | Gene)', fontsize=12)

ax.set_xlabel('Feature'); ax.set_ylabel('Model')

# Colour x-tick labels by type
for tick, feat in zip(ax.get_xticklabels(), col_order):
    tick.set_color('#E87722' if feat in CLINICAL_FEATURES else '#1a1a1a')
    tick.set_fontsize(8)

plt.tight_layout()
plt.savefig('V15_feat_importance_heatmap_all.png', dpi=150, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────
# PLOT D: Per-pipeline importance — clinical vs gene
# ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(PIPELINES), figsize=(7 * len(PIPELINES), 5))
for pi, pipe in enumerate(PIPELINES):
    sub = (imp_df[imp_df.Pipeline == pipe]
           .groupby(['Feature','Feature_Type'])['AUC_Drop'].mean()
           .reset_index()
           .sort_values('AUC_Drop', ascending=False))
    colors_p = ['#E87722' if t == 'Clinical' else '#4C8AC4'
                for t in sub['Feature_Type']]
    axes[pi].bar(sub['Feature'], sub['AUC_Drop'], color=colors_p, edgecolor='white')
    axes[pi].set_title(f'{pipe}', fontsize=10)
    axes[pi].set_xlabel('Feature'); axes[pi].set_ylabel('Mean AUC Drop')
    axes[pi].tick_params(axis='x', rotation=45, labelsize=7)
    axes[pi].grid(axis='y', alpha=0.3)
    axes[pi].legend(handles=[Patch(color='#4C8AC4', label='Gene'),
                               Patch(color='#E87722', label='Clinical')],
                     fontsize=7)
plt.suptitle('Feature Importance Per Pipeline (Gene + Clinical)', fontsize=13)
plt.tight_layout()
plt.savefig('V15_feat_importance_per_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────
# Summary table
# ─────────────────────────────────────────────────────────────────
print("\nOVERALL FEATURE IMPORTANCE RANKING (mean AUC drop across all models & pipelines)")
print("="*65)
for i, row in mean_imp.iterrows():
    tag = '[CLINICAL]' if row['Feature_Type'] == 'Clinical' else '[GENE]   '
    print(f"  {i+1:2d}. {tag}  {row['Feature']:20s}  {row['AUC_Drop']:.4f}")

print(f"\nClinical features summary:")
for _, row in clin_imp.iterrows():
    print(f"  {row['Feature']:20s}: {row['AUC_Drop']:.4f}")

print(f"\nTop-5 gene features:")
for _, row in gene_imp.head(5).iterrows():
    print(f"  {row['Feature']:20s}: {row['AUC_Drop']:.4f}")

## 25. GAT & MOGAT Attention Weights

**HeteroGATv2**: gene→patient GATv2 attention.  
**MOGAT**: same gene→patient GATv2 genomic attention **plus** the fusion gate values showing how much each patient relies on the genomic vs clinical pathway.

In [ ]:
def plot_gene_attention(model, graph, ref_df, title):
    """Visualise gene->patient attention for any model that exposes get_attn_weights().
    For MOGAT the function also prints the fusion gate split (printed inside get_attn_weights).
    """
    model.eval()
    if not hasattr(model, 'get_attn_weights'):
        print(f"  {type(model).__name__} has no get_attn_weights — skipping.")
        return
    try:
        eidx, weights = model.get_attn_weights(graph)
    except Exception as e:
        print(f"  Attention extraction failed: {e}"); return

    gene_ids = eidx[0].cpu().numpy()
    pat_ids  = eidx[1].cpu().numpy()
    w        = weights.cpu().numpy()

    # Mean attention weight per gene (averaged over all edges touching that gene)
    gene_attn = np.zeros(NUM_GENES); gene_cnt = np.zeros(NUM_GENES)
    for g, wt in zip(gene_ids, w):
        if 0 <= g < NUM_GENES:
            gene_attn[g] += wt; gene_cnt[g] += 1
    gene_cnt  = np.maximum(gene_cnt, 1)
    gene_attn /= gene_cnt

    # Top-10 genes and top-15 patients by total received attention
    n_pat = ref_df.shape[0]
    top_g  = np.argsort(gene_attn)[::-1][:10]
    pat_wsum = np.zeros(n_pat)
    for g, p, wt in zip(gene_ids, pat_ids, w):
        if 0 <= p < n_pat: pat_wsum[p] += wt
    top_p = np.argsort(pat_wsum)[::-1][:15]

    mat = np.zeros((len(top_g), len(top_p)))
    for g_loc, g in enumerate(top_g):
        for p_loc, p in enumerate(top_p):
            mask = (gene_ids == g) & (pat_ids == p)
            if mask.any(): mat[g_loc, p_loc] = w[mask].mean()

    fig, axes = plt.subplots(1, 2, figsize=(16, 4))

    # Bar chart: mean attention per gene
    norm_a = gene_attn / (gene_attn.max() + 1e-9)
    axes[0].bar(gene_columns, gene_attn, color=plt.cm.YlOrRd(norm_a))
    axes[0].set_xticklabels(gene_columns, rotation=45, ha='right', fontsize=8)
    axes[0].set_title(f'Mean Gene→Patient Attention\n({title})')
    axes[0].set_ylabel('Mean Attention Weight'); axes[0].grid(axis='y', alpha=0.3)

    # Heatmap: top genes × top patients
    sns.heatmap(mat, ax=axes[1], cmap='YlOrRd',
                xticklabels=[f'P{p}' for p in top_p],
                yticklabels=[gene_columns[g] for g in top_g],
                cbar_kws={'label': 'Attention Weight'})
    axes[1].set_title(f'Top-10 Genes × Top-15 Patients\n({title})')
    axes[1].set_xlabel('Patient Index'); axes[1].set_ylabel('Gene')

    plt.tight_layout()
    safe = title.replace('/','_').replace(' ','_')
    plt.savefig(f'V15_attn_{safe}.png', dpi=150, bbox_inches='tight')
    plt.show()


# ── MOGAT: also show per-patient fusion gate distribution ─────────
def plot_mogat_fusion_gate(model, graph, ref_df, title):
    """Show the distribution of genomic vs clinical pathway weights
    (the learned soft gate) across all patients."""
    model.eval()
    if not isinstance(model, MOGAT):
        return
    with torch.no_grad():
        e   = graph[('Gene','mutates','Patient')].edge_index
        hpg = F.relu(model.pg(graph['Patient'].x))
        hgg = F.relu(model.gg(graph['Gene'].x))
        hpg_out = F.leaky_relu(model.gat((hgg, hpg), e), 0.2)
        hpc     = model.mlp(F.relu(model.pc(graph['Patient'].x)))
        gate_w  = torch.softmax(model.gate(torch.cat([hpg_out, hpc], -1)), dim=-1)
        gw      = gate_w.cpu().numpy()   # [n_patients, 2]  col0=genomic, col1=clinical

    labels  = ref_df['Grade'].values
    grades  = ['Grade 0', 'Grade 1']
    colors  = ['#4C8AC4', '#E87722']

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Histogram of genomic gate weight
    for grade, color, lbl in zip([0,1], colors, grades):
        mask = (labels == grade)
        axes[0].hist(gw[mask, 0], bins=20, alpha=0.6, color=color, label=lbl, density=True)
    axes[0].set_xlabel('Genomic pathway gate weight'); axes[0].set_ylabel('Density')
    axes[0].set_title('Genomic Gate Weight Distribution'); axes[0].legend(); axes[0].grid(alpha=0.3)

    # Histogram of clinical gate weight
    for grade, color, lbl in zip([0,1], colors, grades):
        mask = (labels == grade)
        axes[1].hist(gw[mask, 1], bins=20, alpha=0.6, color=color, label=lbl, density=True)
    axes[1].set_xlabel('Clinical pathway gate weight'); axes[1].set_ylabel('Density')
    axes[1].set_title('Clinical Gate Weight Distribution'); axes[1].legend(); axes[1].grid(alpha=0.3)

    # Scatter: genomic vs clinical gate per patient, coloured by grade
    for grade, color, lbl in zip([0,1], colors, grades):
        mask = (labels == grade)
        axes[2].scatter(gw[mask, 0], gw[mask, 1], c=color, label=lbl,
                         alpha=0.5, s=20, edgecolors='none')
    axes[2].plot([0,1],[1,0],'k--', alpha=0.3, lw=1)   # gate sums to 1
    axes[2].set_xlabel('Genomic gate'); axes[2].set_ylabel('Clinical gate')
    axes[2].set_title('Pathway Gate per Patient'); axes[2].legend(); axes[2].grid(alpha=0.3)

    plt.suptitle(f'MOGAT Fusion Gate Analysis — {title}', fontsize=12)
    plt.tight_layout()
    safe = title.replace('/','_').replace(' ','_')
    plt.savefig(f'V11_mogat_gate_{safe}.png', dpi=150, bbox_inches='tight')
    plt.show()


# ── Run for HeteroGATv2 and MOGAT across all pipelines ───────────
for pipe in PIPELINES:
    for mname in ['HeteroGATv2', 'MOGAT']:
        m = all_models.get((mname, pipe))
        if m is None: continue
        print(f"\nAttention: {mname} / {pipe}")
        plot_gene_attention(m, test_graph, test_df, f"{mname}/{pipe}")
        if mname == 'MOGAT':
            plot_mogat_fusion_gate(m, test_graph, test_df, f"{pipe}")

## 26. Classification Reports — Best Model Overall

In [ ]:
for ds in ['TCGA Test','CGGA']:
    sub  = results_full[results_full.Dataset==ds]
    best = sub.loc[sub['auc'].idxmax()]
    th   = best['threshold']
    preds  = (best['probs'] >= th).astype(int)
    labels = best['labels']
    print(f"{'='*65}")
    print(f"Best on {ds}: {best['Model']} / {best['Pipeline']}")
    print(f"AUC={best['auc']:.4f}  Threshold={th:.3f}")
    print(classification_report(labels, preds, target_names=['Grade 0','Grade 1']))

## 27. Save Results

In [ ]:
# import os
# os.makedirs('saved_models_v15', exist_ok=True)
# for (mname, pipe), model in all_models.items():
#     fn = f"saved_models_v15/{mname}_{pipe.replace(' ','_')}.pth"
#     torch.save(model.state_dict(), fn)

pd.DataFrame([{'Model': mn, 'Pipeline': pp, 'Threshold': th}
               for (mn, pp), th in all_thresholds.items()]).to_csv('V11_thresholds.csv', index=False)
if 'imp_df' in dir() and not imp_df.empty:
    imp_df.to_csv('V11_feature_importance.csv', index=False)
    print("✓ Exported: V11_feature_importance.csv")

print("✓ Saved model weights to saved_models_v15/")
print("\nFinal AUC Pivot:")
print(results_df.pivot_table(index='Model', columns=['Pipeline','Dataset'],
                              values='AUC', aggfunc='mean').round(4).to_string())